# NVDA 뉴스 LLM 분석 파이프라인
- 본문 노이즈 제거
- Claude API로 번역 · 요약 · 감정분석
- 결과 CSV 저장

In [36]:
import pandas as pd
import json
import re
import time

print('✅ 라이브러리 로드 완료')

✅ 라이브러리 로드 완료


## 1. 설정

In [ ]:
import os
from openai import OpenAI

# ⚠️ 여기에 API 키 입력 또는 환경변수로 설정
os.environ["OPENAI_API_KEY"] = ""

CSV_PATH = r"NVDA_news.csv"          # 입력 파일
OUTPUT_PATH = r"NVDA_news_analyzed.csv"  # 출력 파일
MAX_CONTENT_CHARS = 3000            # 본문 앞 3000자만 사용 (토큰 절약)
SLEEP_BETWEEN_CALLS = 1.0           # API 호출 간격 (초)

client = OpenAI()
print('✅ 설정 완료')

✅ 설정 완료


## 2. 데이터 로드 및 미리보기

In [38]:
df = pd.read_csv(CSV_PATH)
print(f'총 기사 수: {len(df)}건')
print(f'컬럼: {df.columns.tolist()}')
df.head(3)

총 기사 수: 73건
컬럼: ['title', 'date', 'link', 'content']


,title,date,link,content
0,1 Clear Signal That Nvidia's Stock Is Primed t...,"Sun, 15 Mar 2026 03:30:00 GMT",https://news.google.com/rss/articles/CBMimAFBV...,Nvidia's stock is priced at nearly the same le...
1,Here are Friday's biggest analyst calls: Nvidi...,"Fri, 13 Mar 2026 12:30:25 GMT",https://news.google.com/rss/articles/CBMiiwFBV...,Got a confidential news tip? We want to hear f...
2,NVIDIA Corporation $NVDA Stock Holdings Booste...,"Sun, 15 Mar 2026 09:57:32 GMT",https://news.google.com/rss/articles/CBMi7AFBV...,California Public Employees Retirement System ...


## 3. 본문 노이즈 제거

In [39]:
NOISE_PATTERNS = [
    r"Sign up for free newsletters.*",
    r"Got a confidential news tip.*",
    r"© \d{4}.*All rights reserved.*",
    r"Cookie.*Notice.*",
    r"Privacy Policy.*",
    r"This site is now part of.*",
    r"Making the world smarter.*",
    r"Invest better with.*",
    r"\*Average returns.*",
    r"Market data powered by.*",
    r"Data is a real-time snapshot.*",
]

def clean_content(text: str) -> str:
    """노이즈 패턴 제거 후 앞부분만 사용"""
    if not isinstance(text, str):
        return ""
    for pattern in NOISE_PATTERNS:
        text = re.sub(pattern, "", text, flags=re.DOTALL | re.IGNORECASE)
    text = re.sub(r"\s{3,}", "\n\n", text)  # 과도한 공백 정리
    return text.strip()[:MAX_CONTENT_CHARS]

# 테스트
sample = clean_content(df['content'][0])
print('정제 후 본문 샘플:')
print(sample[:500])

정제 후 본문 샘플:
Nvidia's stock is priced at nearly the same level as the broader market.
Nvidia (NVDA 1.58%) has made investors a ton of money over the past few years. If you invested $10,000 into it at the start of 2023, that investment is now worth $125,000. That's an incredible return on investment. While Nvidia won't be able to repeat that growth rate over the next three years, it has what it takes to outperform the market.
I think there's one clear signal investors can't ignore about Nvidia's stock, and th


## 4. LLM 분석 함수

In [42]:
SYSTEM_PROMPT = """당신은 금융 뉴스 분석 전문가입니다.
영어 뉴스 기사를 분석하여 반드시 아래 형식의 JSON만 출력하세요.
다른 텍스트나 마크다운 코드블록 없이 JSON만 출력하세요."""

def analyze_news(title: str, content: str) -> dict:
    """Claude API로 뉴스 분석"""
    user_prompt = f"""다음 뉴스를 분석해주세요.

제목: {title}
본문: {content}

아래 JSON 형식으로만 답하세요:
{{
  "summary_ko": "3줄 이내 한국어 요약",
  "sentiment": "긍정 또는 부정 또는 중립",
  "score": -1.0에서 1.0 사이 감정 점수 (숫자만),
  "keywords": ["키워드1", "키워드2", "키워드3", "키워드4", "키워드5"],
  "impact": "주가에 미치는 영향 한 줄 설명"
}}"""

    try:
        response = client.chat.completions.create(
        model="gpt-4o-mini",  # 저렴한 모델
        max_tokens=1000,
        messages=[
         {"role": "system", "content": SYSTEM_PROMPT},
         {"role": "user", "content": user_prompt}
    ]
)
        raw = response.choices[0].message.content.strip()
        raw = re.sub(r"```json|```", "", raw).strip()
        return json.loads(raw)

    except json.JSONDecodeError:
        print(f"  ⚠️  JSON 파싱 실패: {title[:40]}")
        return {"summary_ko": "", "sentiment": "분석실패", "score": 0.0, "keywords": [], "impact": ""}
    except Exception as e:
        print(f"  ❌ API 오류: {e}")
        return {"summary_ko": "", "sentiment": "오류", "score": 0.0, "keywords": [], "impact": ""}

print('✅ 함수 정의 완료')

✅ 함수 정의 완료


## 5. 단일 기사 테스트 (전체 실행 전 확인용)

In [43]:
# 첫 번째 기사만 테스트
test_title = df['title'][0]
test_content = clean_content(df['content'][0])

result = analyze_news(test_title, test_content)
print(json.dumps(result, ensure_ascii=False, indent=2))

{
  "summary_ko": "Nvidia의 주가는 S&P 500과 유사한 수준으로 저평가되어 있으며, 향후 몇 년간 강력한 성장 가능성이 있다. 투자자들은 현재 가격을 저점으로 삼아 주식을 매수해야 한다고 전문가가 강조하고 있다. AI와 데이터 센터 관련 지출 증가가 Nvidia의 성장을 더욱 촉진할 것이다.",
  "sentiment": "긍정",
  "score": 0.8,
  "keywords": [
    "Nvidia",
    "주가",
    "성장",
    "AI",
    "데이터 센터"
  ],
  "impact": "이 뉴스는 Nvidia 주가에 긍정적인 영향을 미칠 것으로 예상된다."
}


## 6. 전체 파이프라인 실행

In [44]:
results = []

for i, row in df.iterrows():
    print(f"[{i+1}/{len(df)}] {row['title'][:50]}...")

    cleaned = clean_content(row['content'])
    analysis = analyze_news(row['title'], cleaned)

    results.append({
        'title': row['title'],
        'date': row['date'],
        'link': row['link'],
        'content_cleaned': cleaned,
        'summary_ko': analysis.get('summary_ko', ''),
        'sentiment': analysis.get('sentiment', ''),
        'score': analysis.get('score', 0.0),
        'keywords': ', '.join(analysis.get('keywords', [])),
        'impact': analysis.get('impact', ''),
    })

    time.sleep(SLEEP_BETWEEN_CALLS)

print('\n🎉 분석 완료!')

[1/73] 1 Clear Signal That Nvidia's Stock Is Primed to Sk...
[2/73] Here are Friday's biggest analyst calls: Nvidia, T...
[3/73] NVIDIA Corporation $NVDA Stock Holdings Boosted by...
[4/73] What's Going On With Nvidia Stock Thursday? - NVID...
[5/73] The Best Stocks to Invest $1,000 In Right Now -- a...
[6/73] Nvidia Stock: Buy, Sell, or Hold? - The Motley Foo...
[7/73] Dow Jones Futures Rise As Oil Prices Hits Resistan...
[8/73] Stock-Split Follow-up: How Nvidia, Alphabet, Amazo...
[9/73] NVIDIA Announces Financial Results for Fourth Quar...
[10/73] Billionaires David Tepper and Michael Platt Sold N...
[11/73] Best Artificial Intelligence (AI) Stock to Buy Now...
[12/73] Stock Market Falls In Volatile Week As Oil Prices ...
[13/73] Why Nvidia Could Remain the Most Important Stock o...
[14/73] Billionaires David Tepper and Michael Platt Sold N...
[15/73] Nvidia's Top AI Event Is Here: Will Nvidia Stock R...
[16/73] Stock Market Today, March 13: Nvidia Slips as GTC ...
[17/73] Here's Wh

## 7. 결과 저장 및 통계

In [45]:
result_df = pd.DataFrame(results)
result_df.to_csv(OUTPUT_PATH, index=False, encoding='utf-8-sig')
print(f'✅ 저장 완료: {OUTPUT_PATH}')

print('\n📊 감정 분포:')
print(result_df['sentiment'].value_counts())
print(f'\n평균 감정 점수: {result_df["score"].mean():.3f}')

result_df[['title', 'date', 'sentiment', 'score', 'summary_ko']].head(10)

✅ 저장 완료: NVDA_news_analyzed.csv

📊 감정 분포:
sentiment
긍정    44
부정    22
중립     7
Name: count, dtype: int64

평균 감정 점수: 0.253


,title,date,sentiment,score,summary_ko
0,1 Clear Signal That Nvidia's Stock Is Primed t...,"Sun, 15 Mar 2026 03:30:00 GMT",긍정,0.9,"Nvidia의 주가는 현재 시장 평균과 유사한 수준에서 거래되고 있으며, 향후 몇 ..."
1,Here are Friday's biggest analyst calls: Nvidi...,"Fri, 13 Mar 2026 12:30:25 GMT",긍정,0.7,금요일에 발표된 주요 애널리스트의 투자 의견과 주목할 만한 기업 목록이 포함되어 있...
2,NVIDIA Corporation $NVDA Stock Holdings Booste...,"Sun, 15 Mar 2026 09:57:32 GMT",긍정,0.7,"캘리포니아 공무원 은퇴 시스템이 NVIDIA의 지분을 2.1% 증가시켰고, 여러 기..."
3,What's Going On With Nvidia Stock Thursday? - ...,"Thu, 12 Mar 2026 16:59:00 GMT",중립,0.0,Nvidia 주식에 대한 뉴스가 오류로 인해 제공되지 않았습니다. 시장 소식과 투자...
4,"The Best Stocks to Invest $1,000 In Right Now ...","Sun, 15 Mar 2026 01:15:00 GMT",긍정,0.8,NVDA와 AVGO는 반도체 분야에서 AI의 급격한 성장으로 큰 성장 잠재력을 보이...
5,"Nvidia Stock: Buy, Sell, or Hold? - The Motley...","Fri, 13 Mar 2026 21:21:00 GMT",긍정,0.7,"Nvidia의 매출 성장과 이익률이 확대되고 있으나, 시장의 주가 평가가 불확실성을..."
6,Dow Jones Futures Rise As Oil Prices Hits Resi...,"Mon, 16 Mar 2026 00:17:10 GMT",중립,0.0,"다우존스 선물이 상승하고 있으며, 유가가 100달러에서 저항에 부딪히고 있습니다. ..."
7,"Stock-Split Follow-up: How Nvidia, Alphabet, A...","Sat, 14 Mar 2026 13:20:00 GMT",부정,-0.7,"전 세계 유가 상승으로 인해 주식 시장이 하락세를 보이고 있으며, 소비자 신뢰도 저..."
8,NVIDIA Announces Financial Results for Fourth ...,"Wed, 25 Feb 2026 08:00:00 GMT",긍정,0.8,NVIDIA는 2026 회계연도 4분기에 681억 달러의 기록적인 매출을 보고하며 ...
9,Billionaires David Tepper and Michael Platt So...,"Sun, 15 Mar 2026 22:47:46 GMT",긍정,0.7,"데이비드 테퍼와 마이클 플랫은 엔비디아 주식을 매도하고 IPO 이후 40,000% ..."
